![](Images/banner_health_analytics.png){fig-align="center" width="100%"}

### Health Analytics with Python · Module 07: Geospatial & Spatial Epidemiology
***
**Learning objectives**
- Understand spatial data types: points, polygons, rasters, and their health applications
- Work with `geopandas` for spatial data manipulation
- Create publication-quality choropleth maps with quintile and Jenks classification
- Apply coordinate reference systems (CRS) and spatial joins
- Map health outcomes with appropriate colour schemes and legends
- Compute and visualise age-standardised mortality rates (SMR)

**Estimated time:** 2.5 hours | **Level:** Intermediate  
**Libraries:** `geopandas`, `matplotlib`, `numpy`, `pandas`


## 1. Setup & Synthetic County Health Dataset

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
warnings.filterwarnings("ignore"); import os; os.makedirs("/tmp/mod07", exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})
np.random.seed(42)

# 200 synthetic counties arranged on a 10x20 grid (approx US-like layout)
NROW, NCOL = 10, 20; N_COUNTIES = NROW * NCOL
row_idx = np.repeat(np.arange(NROW), NCOL)
col_idx = np.tile(np.arange(NCOL), NROW)

# Geographic coordinates (lon/lat-like)
lon = -120 + col_idx * 2.5 + np.random.normal(0, 0.3, N_COUNTIES)
lat =   28 + row_idx * 2.0 + np.random.normal(0, 0.2, N_COUNTIES)

# Socioeconomic and demographic covariates
pct_poverty   = 10 + 8*np.random.beta(2,5,N_COUNTIES) + 3*np.sin(lon/20)
pct_uninsured = 8  + 6*np.random.beta(2,4,N_COUNTIES) + 0.3*pct_poverty
pct_black     = 5  + 20*np.random.beta(1,4,N_COUNTIES)
pct_rural     = 30 + 40*np.random.beta(3,3,N_COUNTIES)
pm25          = 8  + 4*np.random.gamma(2,1,N_COUNTIES) + 0.05*pct_rural
median_income = 45000 - 800*pct_poverty + np.random.normal(0,3000,N_COUNTIES)
population    = np.random.lognormal(10.5, 1.0, N_COUNTIES).astype(int)

# Spatial autocorrelation via spatial smoothing
def spatial_smooth(values, row_idx, col_idx, nrow, ncol, sigma=1.5):
    from scipy.ndimage import gaussian_filter
    grid = np.zeros((nrow, ncol))
    for r,c,v in zip(row_idx, col_idx, values):
        grid[r,c] = v
    return gaussian_filter(grid, sigma=sigma)[row_idx, col_idx]

# Spatially autocorrelated latent risk factor
spatial_risk = spatial_smooth(np.random.normal(0,1,N_COUNTIES), row_idx, col_idx, NROW, NCOL, sigma=2)

# Outcome: age-adjusted CVD mortality rate (per 100k)
TRUE_POVERTY_EFFECT = 1.2  # each 1pp poverty -> 1.2 extra deaths per 100k
cvd_rate = (180 + TRUE_POVERTY_EFFECT*pct_poverty + 0.8*pct_uninsured
            + 0.5*pm25 + 15*spatial_risk + np.random.normal(0, 12, N_COUNTIES))

# Observed deaths and expected (for SMR/SIR)
expected_deaths = (cvd_rate/100000) * population * 5  # 5-year count
observed_deaths = np.random.poisson(expected_deaths).astype(int)
national_rate   = cvd_rate.mean() / 100000
expected_std    = national_rate * population * 5
smr             = observed_deaths / expected_std.clip(1)

county_id = [f"C{i:03d}" for i in range(N_COUNTIES)]
df = pd.DataFrame({
    "county_id":county_id,"lon":lon,"lat":lat,
    "row":row_idx,"col":col_idx,
    "pct_poverty":pct_poverty,"pct_uninsured":pct_uninsured,
    "pct_black":pct_black,"pct_rural":pct_rural,
    "pm25":pm25,"median_income":median_income,"population":population,
    "cvd_rate":cvd_rate,"expected_deaths":expected_deaths,
    "observed_deaths":observed_deaths,"smr":smr,
    "spatial_risk":spatial_risk,
})
print(f"County dataset: N={N_COUNTIES} | CVD rate: {cvd_rate.mean():.1f}/100k (SD={cvd_rate.std():.1f})")
print(f"Poverty: {pct_poverty.mean():.1f}% | Uninsured: {pct_uninsured.mean():.1f}% | PM2.5: {pm25.mean():.1f}")


## 2. Spatial Data Structures in Health Analytics

| Type | Examples | Python tool |
|------|----------|-------------|
| **Points** | Hospital locations, patient addresses, monitoring stations | `geopandas` Point |
| **Polygons** | Counties, census tracts, ZIP codes, health service areas | `geopandas` Polygon |
| **Rasters** | Satellite imagery, PM2.5 grids, land use | `rasterio`, `numpy` |
| **Lines** | Roads, rivers, transit routes | `geopandas` LineString |
| **Grids** | Aggregated kernel density, interpolated surfaces | `scipy.interpolate` |

**Key CRS concepts:**
- WGS 84 (EPSG:4326): latitude/longitude in degrees — for global data
- NAD83 (EPSG:4269): North American datum — for US health data
- Albers Equal Area (EPSG:5070): preserves area — for choropleth maps


In [ ]:
try:
    import geopandas as gpd
    from shapely.geometry import Point, Polygon, box
    GEOPANDAS_OK = True
    print(f"geopandas {gpd.__version__} available")
except ImportError:
    GEOPANDAS_OK = False
    print("pip install geopandas shapely")

if GEOPANDAS_OK:
    # Create GeoDataFrame with synthetic county centroids + bounding boxes
    from shapely.geometry import box as shapely_box
    # Each county is a 2.5° x 2.0° rectangle
    geoms = [shapely_box(lo-1.25, la-1.0, lo+1.25, la+1.0)
             for lo, la in zip(df.lon, df.lat)]
    gdf = gpd.GeoDataFrame(df, geometry=geoms, crs="EPSG:4326")
    print(f"GeoDataFrame: {len(gdf)} counties | CRS: {gdf.crs}")
    print(f"Bounding box: {gdf.total_bounds.round(2)}")
else:
    print("Proceeding with matplotlib scatter-based maps (no geopandas)")


## 3. Choropleth Map — CVD Mortality Rate

In [ ]:
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from mpl_toolkits.axes_grid1 import make_axes_locatable

def scatter_choropleth(ax, lon, lat, values, title, cmap="RdYlGn_r",
                       label="Rate per 100k", n_quantiles=5, size=200):
    """Scatter-based choropleth using quantile classification."""
    quantile_labels = pd.qcut(values, n_quantiles, labels=False, duplicates="drop")
    norm = mcolors.Normalize(vmin=values.min(), vmax=values.max())
    sc = ax.scatter(lon, lat, c=values, cmap=cmap, norm=norm,
                    s=size, alpha=0.85, edgecolors="white", linewidth=0.3)
    plt.colorbar(sc, ax=ax, label=label, shrink=0.8)
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
    ax.set_title(title, fontsize=10)
    return sc

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
# CVD mortality rate
scatter_choropleth(axes[0,0], df.lon, df.lat, df.cvd_rate,
                   "CVD Mortality Rate (per 100k)", "RdYlGn_r", "Rate/100k")
# Poverty rate
scatter_choropleth(axes[0,1], df.lon, df.lat, df.pct_poverty,
                   "Poverty Rate (%)", "YlOrRd", "Poverty (%)")
# SMR
scatter_choropleth(axes[1,0], df.lon, df.lat, df.smr,
                   "Standardised Mortality Ratio (SMR)", "RdBu_r", "SMR", size=200)
# PM2.5
scatter_choropleth(axes[1,1], df.lon, df.lat, df.pm25,
                   "PM2.5 (µg/m³)", "YlOrBr", "PM2.5")
# Reference line SMR=1
axes[1,0].set_title("SMR > 1 = excess mortality | < 1 = deficit", fontsize=9)

plt.suptitle("County-Level Health Atlas: CVD Mortality & Risk Factors", fontsize=13)
plt.tight_layout()
plt.savefig("/tmp/mod07/choropleth_atlas.png", bbox_inches="tight", dpi=150)
plt.show()
print(f"Counties with SMR > 1.2 (excess mortality): {(df.smr>1.2).sum()} ({(df.smr>1.2).mean()*100:.1f}%)")
print(f"Counties with SMR < 0.8 (deficit mortality): {(df.smr<0.8).sum()} ({(df.smr<0.8).mean()*100:.1f}%)")


## 4. Classification Schemes for Health Maps

In [ ]:
# Compare classification schemes: Equal interval, Quantile, Natural Breaks (Jenks)
from scipy.cluster.vq import kmeans

def natural_breaks(values, k=5):
    """Simplified Jenks natural breaks via k-means on 1D data."""
    vals_1d = values.reshape(-1, 1).astype(float)
    centroids, _ = kmeans(vals_1d, k)
    centroids = np.sort(centroids.flatten())
    breaks = []
    for i in range(len(centroids) - 1):
        breaks.append((centroids[i] + centroids[i+1]) / 2)
    return [values.min()] + breaks + [values.max()]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
cmap = plt.cm.RdYlGn_r
n_classes = 5

for ax, title, bins_fn in [
    (axes[0], "Equal Interval",
     lambda v: np.linspace(v.min(), v.max(), n_classes+1)),
    (axes[1], "Quantile (quintiles)",
     lambda v: np.percentile(v, np.linspace(0, 100, n_classes+1))),
    (axes[2], "Natural Breaks (Jenks)",
     lambda v: natural_breaks(v, n_classes)),
]:
    breaks = bins_fn(df.cvd_rate.values)
    labels = pd.cut(df.cvd_rate, bins=breaks, labels=False, include_lowest=True)
    colors = [cmap(labels.iloc[i]/(n_classes-1)) for i in range(len(df))]
    ax.scatter(df.lon, df.lat, c=colors, s=150, edgecolors="white", linewidth=0.3)
    ax.set_title(f"CVD Rate: {title}", fontsize=10)
    ax.set_xlabel("Lon"); ax.set_ylabel("Lat")
    # Legend
    for k, (lo, hi) in enumerate(zip(breaks[:-1], breaks[1:])):
        ax.plot([], [], "o", color=cmap(k/(n_classes-1)),
                label=f"{lo:.0f}–{hi:.0f}", ms=8)
    ax.legend(fontsize=7, title="Rate/100k", loc="lower right")

plt.suptitle("Classification Scheme Comparison — CVD Mortality", fontsize=12)
plt.tight_layout()
plt.savefig("/tmp/mod07/classification_schemes.png", bbox_inches="tight")
plt.show()


## 5. Age-Standardised Rate Computation

In [ ]:
# Simulate age-stratified deaths for direct standardisation
age_groups = ["<45","45-64","65-74","75+"]
std_pop    = np.array([0.35, 0.30, 0.20, 0.15])  # 2000 US standard population weights

np.random.seed(7)
# True age-specific rates by county
base_rates = np.array([50, 200, 600, 1800])  # per 100k per age group
age_specific_rates = np.outer(df.cvd_rate/200, base_rates)  # shape: (N, 4)
age_specific_rates += np.random.normal(0, 20, age_specific_rates.shape)
age_specific_rates = age_specific_rates.clip(10, 3000)

# Direct standardisation: apply standard pop weights to age-specific rates
asmr = (age_specific_rates * std_pop[np.newaxis,:]).sum(axis=1)

# Indirect standardisation: SMR = observed/expected
df["asmr"] = asmr

fig, ax = plt.subplots(figsize=(10,5))
ax.scatter(df.cvd_rate, df.asmr, alpha=0.5, s=30, color="#4878CF")
m, b = np.polyfit(df.cvd_rate, df.asmr, 1)
xl = np.linspace(df.cvd_rate.min(), df.cvd_rate.max(), 100)
ax.plot(xl, m*xl+b, "r-", lw=2, label=f"r = {np.corrcoef(df.cvd_rate,df.asmr)[0,1]:.3f}")
ax.set_xlabel("Crude CVD rate (per 100k)"); ax.set_ylabel("ASMR (per 100k, direct std)")
ax.set_title("Crude vs Age-Standardised Mortality Rate (Correlation reflects confounding by age structure)")
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig("/tmp/mod07/crude_vs_asmr.png", bbox_inches="tight")
plt.show()

diff = df.asmr - df.cvd_rate
print(f"Counties where ASMR > crude rate (older age structure): {(diff>0).sum()} ({(diff>0).mean()*100:.1f}%)")
print(f"Mean difference (ASMR - crude): {diff.mean():.1f} per 100k")


## 6. Spatial Join — Linking Point Data to Polygons

In [ ]:
# Simulate hospital locations (point data) and join to county
np.random.seed(13)
N_HOSPITALS = 50
hosp_lon = np.random.uniform(df.lon.min(), df.lon.max(), N_HOSPITALS)
hosp_lat = np.random.uniform(df.lat.min(), df.lat.max(), N_HOSPITALS)
hosp_beds = np.random.lognormal(5, 0.7, N_HOSPITALS).astype(int).clip(50, 2000)
hosp_trauma = np.random.binomial(1, 0.3, N_HOSPITALS)

# Assign each hospital to nearest county centroid
from scipy.spatial import cKDTree
county_coords = np.column_stack([df.lon, df.lat])
hosp_coords   = np.column_stack([hosp_lon, hosp_lat])
tree = cKDTree(county_coords)
_, county_idx = tree.query(hosp_coords)

# Aggregate hospital stats by county
df["n_hospitals"]   = 0
df["total_beds"]    = 0
df["has_trauma_ctr"]= 0
for i, c_idx in enumerate(county_idx):
    df.loc[c_idx, "n_hospitals"]    += 1
    df.loc[c_idx, "total_beds"]     += hosp_beds[i]
    df.loc[c_idx, "has_trauma_ctr"] += hosp_trauma[i]
df["has_trauma_ctr"] = (df["has_trauma_ctr"] > 0).astype(int)
df["beds_per_1000"]  = df["total_beds"] / df["population"] * 1000

fig, axes = plt.subplots(1,2,figsize=(14,5))
sc = axes[0].scatter(df.lon, df.lat, c=df.beds_per_1000, cmap="Blues", s=150, alpha=0.85)
axes[0].scatter(hosp_lon, hosp_lat, marker="P", s=60, c="red", zorder=5, label="Hospital")
plt.colorbar(sc, ax=axes[0], label="Beds per 1,000 pop")
axes[0].set_title("Hospital Beds per 1,000 Population"); axes[0].legend(fontsize=9)

# Correlation: beds vs CVD mortality
corr = np.corrcoef(df.beds_per_1000, df.cvd_rate)[0,1]
axes[1].scatter(df.beds_per_1000, df.cvd_rate, alpha=0.5, s=30, color="#D65F5F")
m,b = np.polyfit(df.beds_per_1000, df.cvd_rate, 1)
xl = np.linspace(df.beds_per_1000.min(), df.beds_per_1000.max(), 100)
axes[1].plot(xl, m*xl+b, "k-", lw=2)
axes[1].set_xlabel("Beds per 1,000"); axes[1].set_ylabel("CVD rate per 100k")
axes[1].set_title(f"Hospital Capacity vs CVD Mortality (r={corr:.3f} — ecological correlation only!)")
plt.tight_layout()
plt.savefig("/tmp/mod07/hospital_join.png", bbox_inches="tight")
plt.show()


## 7. Knowledge Check
1. A choropleth map uses quantile classification and shows your county in the highest quintile. A colleague uses equal-interval and shows it in the 4th quintile. Which is "correct"?
2. SMR = 1.45 for a rural county. What does this mean, and what information do you need to know if this is statistically significant?
3. You want to map CVD mortality for census tracts. Your denominator (population) comes from the 2020 Census but your numerator (deaths) covers 2018-2022. What are the key assumptions?
4. The correlation between beds/1,000 and CVD rate is -0.31. A colleague concludes "more hospitals reduce CVD deaths." What fallacy is this?
5. Compute a bivariate choropleth: classify counties into 3x3 cells by both poverty rate and CVD mortality, and colour-code the 9 combinations.


In [ ]:
# Exercise 5 — Bivariate choropleth (3x3 = 9 classes)
poverty_cat = pd.qcut(df.pct_poverty, 3, labels=[0,1,2])
cvd_cat     = pd.qcut(df.cvd_rate,    3, labels=[0,1,2])
bivariate_class = poverty_cat.astype(int) * 3 + cvd_cat.astype(int)

# 9-colour palette (low-low=blue, high-high=red, mixed=purple)
biv_colors = {
    0:"#e8e8e8",1:"#dfb0d6",2:"#be64ac",   # low poverty: light->purple
    3:"#ace4e4",4:"#a5add3",5:"#8c62aa",   # mid poverty
    6:"#5ac8c8",7:"#5698b9",8:"#3b4994",   # high poverty: teal->dark
}
colors_biv = [biv_colors[c] for c in bivariate_class]

fig, ax = plt.subplots(figsize=(10,6))
ax.scatter(df.lon, df.lat, c=colors_biv, s=200, edgecolors="white", linewidth=0.3)
ax.set_title("Bivariate Choropleth: Poverty Rate × CVD Mortality\n(blue=low/low, red=high/high, purple=high poverty/low CVD)")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
# Add legend
for cls, color in biv_colors.items():
    pov_lbl = ["Low","Med","High"][cls//3]
    cvd_lbl = ["Low","Med","High"][cls%3]
    ax.plot([],[],"o",color=color,ms=8,label=f"Pov:{pov_lbl} CVD:{cvd_lbl}")
ax.legend(ncol=3,fontsize=7,loc="lower right")
plt.tight_layout()
plt.savefig("/tmp/mod07/bivariate_choropleth.png",bbox_inches="tight")
plt.show()


***
**Next → NB-02: Spatial Autocorrelation & Moran's I**
